<a href="https://colab.research.google.com/github/faisalkhan4k/RoomMind/blob/main/notebooks/Multi_Agent_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install Hugging Face transformers, acceleration tools, and the Google GenAI SDK
!pip install -q transformers accelerate google-genai pillow num2words

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 14.2 MB/s eta 0:00:00


In [2]:
import os
import torch
from PIL import Image
from google import genai
from google.genai import types
from google.colab import userdata
from transformers import AutoProcessor
from transformers import AutoModelForImageTextToText


# Retrieve your API key securely from Colab Secrets
try:
    api_key = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=api_key)
    print("✅ Gemini API Client initialized successfully!")
except Exception as e:
    print("❌ Error loading Gemini API key. Ensure 'GEMINI_API_KEY' is added to your Colab Secrets.")

✅ Gemini API Client initialized successfully!


In [3]:
print("⏳ Loading RoomMind-SmolVLM2-500M onto GPU...")

# Set up the device (using T4 GPU in free Colab)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load the processor and model from your Hugging Face repository
model_id = "mohammedfaisalkhan4000/RoomMind-SmolVLM2-500M"
processor = AutoProcessor.from_pretrained(model_id)
vlm_model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32
).to(device)

def run_vision_agent(image_path: str) -> str:
    """Agent 1: Extracts structured furniture/room data from an image."""
    image = Image.open(image_path).convert("RGB")

    # 🌟 FIXED: Added the required "<image>" placeholder token
    prompt = "<image>\n<|object_detection|>"

    inputs = processor(text=prompt, images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        generated_ids = vlm_model.generate(**inputs, max_new_tokens=300)

    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return generated_text

print(f"✅ Vision Agent is ready using device: {device.upper()}")

⏳ Loading RoomMind-SmolVLM2-500M onto GPU...


processor_config.json:   0%|          | 0.00/1.46k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/403 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.32k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/867 [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.03G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/135 [00:00<?, ?B/s]

✅ Vision Agent is ready using device: CUDA


In [4]:
WRITER_PERSONA = (
    "You are an expert E-commerce Copywriter specializing in real estate and furniture sales. "
    "Your job is to take raw, structured data about room furniture and write a highly compelling, "
    "engaging, and professional ad description for online marketplaces like Facebook Marketplace or Craigslist."
)

def run_writer_agent(structured_data: str) -> str:
    """Agent 2: Transforms structured data into an engaging ad copy."""
    user_prompt = f"Create a captivating marketplace ad description based on this extracted room data:\n{structured_data}"

    response = client.models.generate_content(
        model='gemini-2.5-flash', # Lightweight, cheap, fast generic text model
        contents=user_prompt,
        config=types.GenerateContentConfig(
            system_instruction=WRITER_PERSONA,
            temperature=0.8,
        ),
    )
    return response.text
print("✅ Copywriter Agent is ready!")

✅ Copywriter Agent is ready!


In [5]:
# 1. Download a sample image to test with (or upload your own to 'room.jpg')
!wget -O room.jpg https://images.unsplash.com/photo-1616486338812-3dadae4b4ace?w=500 -q
print("🖼️ Test image downloaded as 'room.jpg'\n")

def run_multi_agent_pipeline(image_file: str):
    print("🚀 [Orchestrator] Step 1: Sending image to Local Vision Agent...")

    # Run Agent 1 (Your Fine-Tuned Model)
    extracted_features = run_vision_agent(image_file)
    print(f"↳ 🤖 [Vision Agent Output]:\n{extracted_features}\n")

    print("🚀 [Orchestrator] Step 2: Passing features to Gemini Copywriter Agent...")

    # Run Agent 2 (Gemini Text Model using the state passed from Agent 1)
    final_ad_copy = run_writer_agent(extracted_features)

    print("✨ [Orchestrator] Multi-Agent Workflow Complete! Final Ad Output:\n")
    print(final_ad_copy)

# Execute the workflow
run_multi_agent_pipeline("room.jpg")

🖼️ Test image downloaded as 'room.jpg'

🚀 [Orchestrator] Step 1: Sending image to Local Vision Agent...
↳ 🤖 [Vision Agent Output]:





<|object_detection|> ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||

🚀 [Orchestrator] Step 2: Passing features to Gemini Copywriter Agent...
✨ [Orchestrator] Multi-Agent Workflow Complete! Final Ad Output:

**🔥 Elevate Your Sanctuary: Stunning 4-Piece Modern Bedroom Set – Pristine Condition! 🔥**

Transform your personal retreat into a haven of style and comfort with this exquisite modern bedroom furniture set. Designed with clean lines and a sophisticated aesthetic, this collection offers both timeless elegance and practical functionality, making it the perfect foundation for your dream bedroom.

**What